# Minizinc Solver Runner

!!! ATTENTION:
    This file is really memory hungry as minizinc is inneficient with such large models. Use with caution.


## Imports

In [68]:
from minizinc import Instance, Model, Result, Solver
import sys
import json

sys.path.append("../data-structure")
from instance_data import InstanceData
sys.path.append("..")
import checker


## Minizinc Data Preparation

In [69]:
class ZincInstanceData:
    """ZincInstanceData is a class that loads the instance data from a JSON file and provides access to the data."""

    _n: int = 0
    _m: int = 0
    _horizon: int = 0
    _capable: list[list[int]] = []
    _duration: list[list[int]] = []
    _release: list[list[int]] = []
    _setup: list[list[list[int]]] = []

    def __init__(self, raw_instance: InstanceData):
        self.raw_instance = raw_instance
        self._n = raw_instance.json_data["n"]
        self._m = raw_instance.json_data["m"]
        self._horizon = raw_instance.json_data["horizon"]
        self._capable = raw_instance.json_data["capable"]
        self._duration = raw_instance.json_data["duration"]
        self._release = raw_instance.json_data["release"]
        self._setup = raw_instance.json_data["setup"]

    def to_dzn(self) -> str:
        """Convert the instance data to a string in the format of a .dzn file."""

        duration_flattened = [item for sublist in self._duration for item in sublist]
        release_flattened = [item for sublist in self._release for item in sublist]
        setup_flattened = [item for sublist in self._setup for innerlist in sublist for item in innerlist]

        dzn_str = f"n = {self._n};\n"
        dzn_str += f"m = {self._m};\n"
        dzn_str += f"horizon = {self._horizon};\n"
        dzn_str += f"capable = {[set(n) for n in self._capable]};\n"
        dzn_str += f"duration = array2d(1..n, 0..m-1, [{', '.join(str(d) for d in duration_flattened)}]);\n"
        dzn_str += f"release = array2d(1..n, 0..m-1, [{', '.join(str(r) for r in release_flattened)}]);\n"
        dzn_str += f"setup = array3d(1..n, 1..n, 0..m-1, [{', '.join(str(s) for s in setup_flattened)}]);\n"
        return dzn_str

## Minzinc runner

In [70]:
class ZincRunner:
    """ZincRunner is responsible for running the model with the instance data."""

    _model: Model
    _data: ZincInstanceData
    _solver: Solver
    _instance: Instance

    def __init__(self, data: ZincInstanceData, model: str, solver: str = "highs"):
        self._model = Model(model)
        self._data = data
        self._solver = Solver.lookup(solver)

        self._instance = Instance(self._solver, self._model)

        self._load_data()

    def _load_data(self) -> None:
        self._instance.add_string(self._data.to_dzn())

    async def solve(self):
        return await self._instance.solve_async()


## Main execution

In [71]:
def setup(path: str = "../examples/75_3_5_H.json") -> ZincInstanceData:
    """Setup the instance data for the model."""
    raw_instance = InstanceData(path)
    zinc_instance = ZincInstanceData(raw_instance)
    return zinc_instance


async def run():
    file = "../examples/75_3_5_H.json"
    # Currently we running with the small instance for testing purposes, but this can be easily changed to run with any instance by changing the path in the setup function.
    data = setup(file)
    runner = ZincRunner(data, "./model.mzn")
    result = await runner.solve()
    result_json = json.loads(str(result))
    print(json.dumps(result_json, indent=4))
    instance_json = checker.load_json(file)
    feasible, result = checker.check_and_evaluate(instance_json, result_json)
    print("Checker result:")
    print(f"{'feasible' if feasible else 'infeasible'}, makespan: {result}")

await run()

{
    "makespan": 1049,
    "schedule": {
        "0": [],
        "1": [
            5
        ],
        "2": [
            2,
            3,
            1,
            4
        ]
    }
}
Checker result:
feasible, makespan: 1049
